# Evaluating retrieval

Exercise notebook for the lesson **[Evaluating retrieval](https://ramkumarmd.com/medical-ai/rag/retrieval-evaluation)** in the *Retrieval-augmented generation (RAG)* track of the Medical AI course by Dr. Ram Kumar G.

**In short:** If retrieval fails, the model never sees the right evidence, so evaluate retrieval separately from answers. Build a set of realistic questions labelled with the passages that answer them, then measure how often the right passage appears in the top results (recall@k) and how high it ranks (mean reciprocal rank). Compare methods on the same set, and read every miss: small test sets are noisy, and the misses tell you what to fix.

> All data in this notebook are synthetic or openly licensed. Never use real patient data here, and never paste identifiable patient information into public AI tools. Educational use only: the code and models are not medical devices.

## Setup

In [ ]:
%pip install -q sentence-transformers

## Code from earlier lessons

This notebook builds on code from earlier lessons. Run these cells first.

From **[Chunking clinical guidelines](https://ramkumarmd.com/medical-ai/rag/chunking-clinical-guidelines)**: A synthetic clinic protocol with five headed sections.

In [ ]:
PROTOCOL = """# Synthetic diabetes clinic protocol (teaching document, not clinical guidance)

## 1. Monitoring glucose control
Check HbA1c every 3 months after any change in treatment. Once targets are met and treatment is stable, check HbA1c every 6 months. A common target for many non-pregnant adults is below 7%, adjusted for the individual.

## 2. Kidney function and metformin
Check eGFR and urine albumin-to-creatinine ratio at least once a year. Metformin should not be started if eGFR is below 45. If eGFR falls below 30 during treatment, stop metformin.

## 3. Foot care
Examine both feet at every review. Test sensation with a 10 g monofilament at least once a year. Refer urgently if there is an ulcer with spreading redness, a suspected infection of the bone, or a cold, pale foot.

## 4. Low blood glucose
A glucose below 70 mg/dL is low. Treat with 15 g of fast-acting carbohydrate and recheck after 15 minutes. Repeat if glucose is still below 70 mg/dL. Review any medicine that may have caused the low glucose.

## 5. Eye screening
Arrange retinal screening at diagnosis of type 2 diabetes and then at the interval set by the local screening programme.
"""

## Worked example

These cells reproduce the lesson's code. Lines starting with `#` after the code show the output you should see (numbers can differ slightly with other library versions).

### Scoring three retrieval methods

Run the protocol block from the chunking lesson first. Twelve labelled questions; TF-IDF and embedding scores.

In [ ]:
import re
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

sections = [p.strip() for p in re.split(r"\n(?=## )", PROTOCOL)[1:]]
ids = ["glucose", "kidney", "foot", "hypo", "eye"]

tests = [   # (question, id of the section that answers it)
    ("How often should HbA1c be checked after changing treatment?", "glucose"),
    ("What HbA1c target suits most adults?", "glucose"),
    ("My sugar dropped really low, what now?", "hypo"),
    ("What should I do about a hypo?", "hypo"),
    ("How many grams of carbohydrate treat a low?", "hypo"),
    ("Is the diabetes pill safe with weak kidneys?", "kidney"),
    ("When do we need to stop the biguanide?", "kidney"),
    ("How often is urine albumin checked?", "kidney"),
    ("Numb toes test", "foot"),
    ("When is a foot ulcer an emergency?", "foot"),
    ("How often are retinal photos needed?", "eye"),
    ("When should eye screening start?", "eye"),
]
questions = [q for q, _ in tests]
relevant = [ids.index(r) for _, r in tests]

vec = TfidfVectorizer(stop_words="english").fit(sections + questions)
keyword = cosine_similarity(vec.transform(questions), vec.transform(sections))
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
embedding = (model.encode(questions, normalize_embeddings=True)
             @ model.encode(sections, normalize_embeddings=True).T)

### Recall@k and MRR

Continuing from the previous block: metrics for keyword, embedding and a hybrid of both.

In [ ]:
def ranks(scores):
    """1-based rank of the correct section for each question."""
    order = np.argsort(-scores, axis=1)
    return np.array([list(order[i]).index(relevant[i]) + 1 for i in range(len(relevant))])

def report(name, scores):
    r = ranks(scores)
    print(f"{name:<10} recall@1 {np.mean(r <= 1):.2f}   recall@3 {np.mean(r <= 3):.2f}   "
          f"MRR {np.mean(1 / r):.2f}")

def reciprocal_rank_fusion(*score_sets, k=60):
    """Hybrid search: add 1 / (k + rank) from each method."""
    fused = np.zeros_like(score_sets[0])
    for s in score_sets:
        order = np.argsort(-s, axis=1)
        for i in range(s.shape[0]):
            for rank, j in enumerate(order[i], start=1):
                fused[i, j] += 1 / (k + rank)
    return fused

report("Keyword", keyword)
report("Embedding", embedding)
report("Hybrid", reciprocal_rank_fusion(keyword, embedding))

# Keyword    recall@1 0.83   recall@3 0.92   MRR 0.88
# Embedding  recall@1 0.92   recall@3 1.00   MRR 0.94
# Hybrid     recall@1 0.92   recall@3 0.92   MRR 0.94

### Reading the misses

Continuing from the previous block: which questions each method failed to rank first.

In [ ]:
for name, s in [("Keyword", keyword), ("Embedding", embedding),
                ("Hybrid", reciprocal_rank_fusion(keyword, embedding))]:
    r = ranks(s)
    misses = [(questions[i], int(r[i])) for i in np.where(r > 1)[0]]
    print(name, "misses:", misses)

# Keyword misses: [('What should I do about a hypo?', 4), ('Is the diabetes pill safe with weak kidneys?', 3)]
# Embedding misses: [('What should I do about a hypo?', 3)]
# Hybrid misses: [('What should I do about a hypo?', 4)]

## Exercise

**Goal:** Build and use a retrieval test set to choose and improve a retrieval method.

**Data:** Synthetic protocol and labelled questions in the notebook (Created for this course; not clinical guidance)

### Step 1

Reproduce the three methods' recall@1, recall@3 and MRR.

In [ ]:
# Your code here


### Step 2

Add twelve more questions, including three the protocol cannot answer, and decide how to score 'no answer' questions.

In [ ]:
# Your code here


### Step 3

Apply the abbreviation expansion from the embeddings lesson and re-score all methods.

In [ ]:
# Your code here


### Step 4

Bootstrap the question set (resample with replacement 1,000 times) to get confidence intervals for recall@1 and see whether the methods really differ.

In [ ]:
# Your code here


### Step 5

Implement nDCG@3 with graded labels (2 = answers fully, 1 = partly relevant) and compare rankings.

In [ ]:
# Your code here


---

Lesson: [https://ramkumarmd.com/medical-ai/rag/retrieval-evaluation](https://ramkumarmd.com/medical-ai/rag/retrieval-evaluation). Code in this notebook is released under the MIT licence; the text under CC BY 4.0. See the repository README for details.